<a href="https://colab.research.google.com/github/gemstarchanti/create/blob/master/ImagetoSTL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import sys
import importlib

# Define the path to the TripoSR main directory
triposr_root_path = '/content/TripoSR'

# Clone TripoSR if it doesn't exist
if not os.path.exists(triposr_root_path):
    # Ensure we are in /content before cloning to place TripoSR correctly
    current_dir_before_clone = os.getcwd()
    if current_dir_before_clone != '/content':
        %cd /content
    !git clone https://github.com/VAST-AI-Research/TripoSR.git

# Change directory to TripoSR for installing requirements and running other commands
%cd {triposr_root_path}

# Install requirements
!pip install -r requirements.txt

# Install onnxruntime explicitly, as it's a dependency of rembg and might not be pulled automatically
!pip install onnxruntime

# Install pymeshlab
!pip install pymeshlab

# Explicitly install a compatible version of huggingface-hub
# transformers==4.35.0 requires huggingface-hub<1.0,>=0.16.4 and uses functionality from >=0.19.0
# accelerate 1.12.0 requires huggingface_hub>=0.21.0
!pip install huggingface_hub==0.21.0 --force-reinstall

# Add TripoSR's root directory to sys.path for module discovery
# This ensures that 'tsr' module within TripoSR is discoverable
if triposr_root_path not in sys.path:
    sys.path.insert(0, triposr_root_path)

# Create an empty __init__.py in the tsr directory if it doesn't exist
# This ensures Python recognizes 'tsr' as a package.
ts_init_path = os.path.join(triposr_root_path, 'tsr', '__init__.py')
if not os.path.exists(ts_init_path):
    print(f"[DEBUG] Creating empty __init__.py at: {ts_init_path}")
    with open(ts_init_path, 'w') as f:
        pass

# Debugging statements
print(f"[DEBUG] Current Working Directory: {os.getcwd()}")
print(f"[DEBUG] sys.path contains: {sys.path}")
print(f"[DEBUG] Path to tsr/system.py: {os.path.join(triposr_root_path, 'tsr', 'system.py')}")
print(f"[DEBUG] Does tsr/system.py exist? {os.path.exists(os.path.join(triposr_root_path, 'tsr', 'system.py'))}")
print(f"[DEBUG] Does tsr/__init__.py exist? {os.path.exists(ts_init_path)}")

# Invalidate import caches (keep this as it can help with fresh imports)
importlib.invalidate_caches()

# Removed: del sys.modules['tsr'] and del sys.modules['tsr.system'] as they might be causing problems for subsequent cells.
# Removed: Direct imports of torch, time, PIL, numpy, TSR, etc. as they should be done in the cells that need them.

/content/TripoSR
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-0tfanpu8
  Running command git clone --filter=blob:none --quiet https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-0tfanpu8
  Resolved https://github.com/tatsy/torchmcubes.git to commit 3381600ddc3d2e4d74222f8495866be5fafbace4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.17.3-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.17.3-py3-none-any.whl (295 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.21.0
    Uninstalling huggingface-hub-0.21.0:
      Successfully uninstalled huggingface-hub-0.21.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires hu

[DEBUG] Current Working Directory: /content/TripoSR
[DEBUG] sys.path contains: ['/content/TripoSR', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
[DEBUG] Path to tsr/system.py: /content/TripoSR/tsr/system.py
[DEBUG] Does tsr/system.py exist? True
[DEBUG] Does tsr/__init__.py exist? True


In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
from google.colab import files
from PIL import Image
uploaded = files.upload()
original_image = Image.open(list(uploaded.keys())[0])
original_image.resize((512, 512)).save("examples/product.png")

Saving c0464c7b-a054-4064-b8af-24fc00bf2977-removebg-preview.png to c0464c7b-a054-4064-b8af-24fc00bf2977-removebg-preview (1).png


In [8]:
import torch
import time
from tsr.system import TSR

class Timer:
    def __init__(self):
        self.items = {}
        self.time_scale = 1000.0  # ms
        self.time_unit = "ms"
    def start(self, name: str) -> None:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.items[name] = time.time()
    def end(self, name: str) -> float:
        if name not in self.items:
            return
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = self.items.pop(name)
        delta = time.time() - start_time
        t = delta * self.time_scale
        print(f"{name} finished in {t:.2f}{self.time_unit}.")
timer = Timer()

# Define the missing variables
pretrained_model_name_or_path = "stabilityai/TripoSR"
chunk_size = 2048 # A common default value

timer.start("Initializing model")
model = TSR.from_pretrained(
    pretrained_model_name_or_path,
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(chunk_size)
model.to(device)
timer.end("Initializing model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.yaml:   0%|          | 0.00/987 [00:00<?, ?B/s]

model.ckpt:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


config.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Initializing model finished in 41659.72ms.


In [10]:
import rembg
import numpy as np
import os
from tsr.utils import remove_background, resize_foreground

timer.start("Processing images")
images = []

# Define missing variables
foreground_ratio = 0.85  # Common default value for foreground ratio
output_dir = "output"  # Define output directory
os.makedirs(output_dir, exist_ok=True)

rembg_session = rembg.new_session()
image = remove_background(original_image, rembg_session)
image = resize_foreground(image, foreground_ratio)

if image.mode == "RGBA":
    image = np.array(image).astype(np.float32) / 255.0
    image = image[:, :, :3] * image[:, :, 3:4] + (1 - image[:, :, 3:4]) * 0.5
    image = Image.fromarray((image * 255.0).astype(np.uint8))

image_dir = os.path.join(output_dir, str(0))
os.makedirs(image_dir, exist_ok=True)
image.save(os.path.join(image_dir, "input.png"))
images.append(image)
timer.end("Processing images")

100%|████████████████████████████████████████| 176M/176M [00:00<00:00, 163GB/s]


Processing images finished in 4635.05ms.


In [14]:
import os # Added os import for os.path.join
from tsr.utils import save_video # Import save_video

render = True  # Set to False if you don't want to render videos and images
model_save_format = "obj" # or "glb"

for i, image in enumerate(images):
    print(f"Running image {i + 1}/{len(images)} ...")
    timer.start("Running model")
    with torch.no_grad():
        scene_codes = model([image], device=device)
    timer.end("Running model")

    if render:
        timer.start("Rendering")
        render_images = model.render(scene_codes, n_views=30, return_type="pil")
        for ri, render_image in enumerate(render_images[0]):
            render_image.save(os.path.join(output_dir, str(i), f"render_{ri:03d}.png"))
        save_video(
            render_images[0], os.path.join(output_dir, str(i), "render.mp4"), fps=30
        )
        timer.end("Rendering")

    timer.start("Exporting mesh")
    meshes = model.extract_mesh(scene_codes, has_vertex_color=False)
    mesh_file = os.path.join(output_dir, str(i), f"mesh.{model_save_format}")
    meshes[0].export(mesh_file)
    timer.end("Exporting mesh")
print("Processing complete.")

Running image 1/1 ...
Running model finished in 71843.20ms.
Rendering finished in 1521473.78ms.
Exporting mesh finished in 101965.52ms.
Processing complete.


In [16]:
import pymeshlab

obj_file = "/content/TripoSR/output/0/mesh.obj"
# Load the .obj mesh
ms = pymeshlab.MeshSet()
ms.load_new_mesh(obj_file)
mesh = ms.current_mesh()
# Convert to .stl format
stl_file = 'model.stl'
ms.save_current_mesh(stl_file)